In [1]:
import cv2

cap = cv2.VideoCapture(0)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    
    ret, frame = cap.read()
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    cv2.imshow('frame', gray)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()

## 1) La raspberry

In [2]:
import cv2

IP_RASPBERRY = "192.168.1.112"
PUERTO = 8000
URL_STREAM = f"http://{IP_RASPBERRY}:{PUERTO}/stream.mjpg"

cap = cv2.VideoCapture(URL_STREAM)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    
    ret, frame = cap.read()
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    cv2.imshow('frame', gray)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()

## 2) Método seguro

In [3]:
from camarapi import CamaraPi

In [4]:
cam = CamaraPi()

while True:
    ret, frame = cam.read()
    if not ret:
        continue
    
    # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow("frame", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

In [5]:
from writerpi import WriterPi
import time

fps_objetivo = 30
periodo = 1 / fps_objetivo

cam = CamaraPi()
tam = (cam.width, cam.height)

esc = WriterPi('mysupervideo.mp4', 
                cv2.VideoWriter_fourcc(*'DIVX'),
                fps_objetivo, 
                tam)

while True:
    inicio = time.time()
    
    ret, frame = cam.read()
    if not ret:
        continue
    
    # Operations (drawings)
    esc.write(frame.copy())
    
    cv2.imshow('frame', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        
        break
    
    # Esperar para los fps definidos
    dt = time.time() - inicio
    if dt < periodo:
        time.sleep(periodo - dt)
    
cam.release()
esc.close()
cv2.destroyAllWindows()
    

# Drawing on Live Camera

In [2]:
import cv2
from camarapi import CamaraPi

cam = CamaraPi()

tam = (int(cam.width), int(cam.height))

# TOP LEFT CORNER
x = tam[0] // 2
y = tam[1] // 3

# width and height of RECTANBLE
w = tam[0] // 4
h = tam[1] // 4

# BOTTOM RIGHT x+w, y+h

while True:
    
    ret, frame = cam.read()
    
    cv2.rectangle(frame, (x, y), (x+w, y+h), color=(192, 145, 210), thickness=3)
    
    cv2.imshow('frame', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()

In [1]:
import cv2
from camarapi import CamaraPi



## CALLBACK FUNCTION
def draw_rectangle(event, x, y, flags, param):
    global pt1, pt2, topLeft_clicked, botRight_clicked
    
    if event == cv2.EVENT_LBUTTONDOWN:
        
        # RESET THE RECTANGLE
        if topLeft_clicked == True and botRight_clicked == True:
            pt1 = (0, 0)
            pt2 = (0, 0)
            topLeft_clicked = False
            botRight_clicked = False
            
        if topLeft_clicked == False:
            pt1 = (x, y)
            topLeft_clicked = True
        
        elif botRight_clicked == False:
            pt2 = (x, y)
            botRight_clicked = True


## GLOBAL VARIABLES
pt1 = (0, 0)
pt2 = (0, 0)
topLeft_clicked = False
botRight_clicked = False

# CONNECT TO THE CALLBACK
cam = CamaraPi()
cv2.namedWindow('Test')
cv2.setMouseCallback('Test', draw_rectangle)

while True:
    
    ret, frame = cam.read()
    
    # DRAWING ON THE FRAME BASED OFF THE GLOBAL VAIABLES
    if topLeft_clicked:
        cv2.circle(frame, center=pt1,radius=3, color=(0, 0, 255), thickness=-1)
        
    if topLeft_clicked and botRight_clicked:
        cv2.rectangle(frame, pt1, pt2, (146, 136, 96), 3)
    
    cv2.imshow('Test', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cam.release()
cv2.destroyAllWindows()